# Intro 3: text embeddings with model2vec

In preprocessing we need to decide which parts of the graph schema a question is about. We do it with embeddings: the question and every schema element are turned into vectors, and then we compare the vectors. The library is `model2vec`, which uses a *static* model. Each token has one fixed vector and a text is the average of its tokens, so there is no neural network to run and we need neither a GPU nor an API key.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from model2vec import StaticModel

model = StaticModel.from_pretrained("minishlab/potion-base-8M")
model.embedding.shape

(29528, 256)

The whole model is this one table, a 256-dimensional vector for each of roughly 30 thousand tokens. The first run downloads it from Hugging Face.

## From text to vectors

`encode` takes a list of texts and returns one row per text.

In [2]:
texts = ["Which compounds treat multiple sclerosis?",
         "compound treats disease",
         "compound binds gene",
         "gene participates in pathway"]
vectors = model.encode(texts)
vectors.shape

(4, 256)

## Comparing texts

The usual measure is cosine similarity. These vectors already have length 1, so it is just the dot product, and one matrix product gives all pairs at once.

In [3]:
print("lengths:", np.linalg.norm(vectors, axis=1).round(3))
pd.DataFrame(vectors @ vectors.T, index=texts, columns=range(len(texts))).round(2)

lengths: [1. 1. 1. 1.]


,0,1,2,3
Which compounds treat multiple sclerosis?,1.00,0.62,0.34,0.05
compound treats disease,0.62,1.00,0.45,0.05
compound binds gene,0.34,0.45,1.00,0.34
gene participates in pathway,0.05,0.05,0.34,1.00


The question in the first row is closest to *compound treats disease*, which is the relationship it asks about, and it has almost nothing in common with the sentence about pathways.

## What "static" means

We can rebuild a vector by hand. Split the text into tokens, look up their vectors in the table, take the mean and scale it to length 1. The result is what `encode` returns.

In [4]:
text = "compound treats disease"
tokens = model.tokenizer.encode(text, add_special_tokens=False)
print(tokens.tokens)

by_hand = model.embedding[tokens.ids].mean(axis=0)
by_hand = by_hand / np.linalg.norm(by_hand)
np.allclose(by_hand, model.encode([text])[0], atol=1e-5)

['compound', 'treats', 'disease']


True

Two things follow from this, and both are good for us. The same text always gives exactly the same vector, which is what we want in a project about repeatability. And encoding is fast even on a laptop.

In [5]:
import time

print("same vectors on a second call:", np.array_equal(model.encode(texts), model.encode(texts)))

start = time.time()
model.encode([f"compound number {i} treats disease number {i}" for i in range(1000)])
print(f"1000 short texts in {time.time() - start:.2f} s")

same vectors on a second call: True
1000 short texts in 0.01 s


## Ranking candidates for a question

This is how we use embeddings in the project. We take a question and a list of candidate descriptions and sort the candidates by similarity to the question.

In [6]:
question = "Which drugs relieve the symptoms of asthma?"
candidates = ["compound treats disease", "compound palliates disease", "compound causes side effect",
              "disease presents symptom", "gene interacts with gene"]

scores = model.encode(candidates) @ model.encode([question])[0]
pd.DataFrame({"candidate": candidates, "similarity": scores.round(2)}).sort_values("similarity", ascending=False)

,candidate,similarity
0,compound treats disease,0.40
2,compound causes side effect,0.35
3,disease presents symptom,0.30
1,compound palliates disease,0.18
4,gene interacts with gene,0.00


Here the method shows its weak side. The relationship we wanted is *compound palliates disease*, because to palliate means to relieve symptoms, and it ends up fourth out of five. The model compares words and does not know that "relieve the symptoms" and "palliates" say the same thing.

We still use it, because it is simple, fast and gives the same result every time. But the preprocessing notebook does not take the ranking on trust and measures how often it puts the right schema elements at the top for our questions.